# 🧠 Notebook: Retrieval-Augmented Generation (RAG) — *Part 3 of 3*

*This is the third of three notebooks in this chapter: 1) 2-Step RAG → 2) Agentic RAG → 3) **Hybrid RAG**. Start with `07_1_two_step_rag.ipynb` if you haven't yet — this notebook reuses its knowledge base and embedding cache.*

## 📚 Sources

- [LangChain Documentation: Retrieval](https://docs.langchain.com/oss/python/langchain/retrieval)
- [LangChain Documentation: Build a Custom RAG Agent with LangGraph](https://docs.langchain.com/oss/python/langgraph/rag-agent)

## From Agentic to Hybrid RAG

Recap of where we are:

- **2-Step RAG** always retrieves, and never checks whether the retrieval was any good.
- **Agentic RAG** lets the LLM decide *whether* to retrieve, but once it does, it still has no structured way to validate the result — if the retrieved tweets are irrelevant, or the generated answer is wrong, the agent has no explicit mechanism to notice and fix that.

**Hybrid RAG** combines a structured, predictable pipeline (like 2-Step) with validation steps that can loop back and self-correct (a bit like an agent, but with explicit, inspectable control flow instead of leaving every decision to the LLM's free-form reasoning):

| Validation step | Question it asks | Action if it fails |
|---|---|---|
| **Retrieval validation** | Are the retrieved documents actually relevant to the question? | Rewrite the question and retrieve again |
| **Query enhancement** | Is the question well-formed enough to retrieve well? | Rewrite it to be clearer / more specific |
| **Answer validation** | Does the generated answer actually address the question? | Rewrite the question and try again |

This is often called **Corrective RAG**. We'll build it as an explicit graph, so every step and every decision is visible and debuggable — a big advantage over an opaque agent loop when reliability matters (e.g. an answer that must be grounded in real tweets, not "close enough").

In [13]:
import os
from dotenv import load_dotenv

load_dotenv()

LLM_HOST = os.environ["LLM_HOST"]
LLM_URL = f"http://{LLM_HOST}:11434"
LLM_REASONING = "gemma4:26b"
EMBEDDING_MODEL = "embeddinggemma"

## Rebuilding the knowledge base

Same corpus and cache as the previous two notebooks — see `07_1_two_step_rag.ipynb` for the detailed explanation of `Document`s, chunking, and the embedding cache.

In [14]:
import json
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

with open("content/airline_tweets.json") as f:
    tweets = json.load(f)

docs = [Document(page_content=t["text"], metadata={"sentiment": t["sentiment"]}) for t in tweets]

CACHE_PATH = "content/embedding_cache.json"


class CachedOllamaEmbeddings(OllamaEmbeddings):
    # See notebook 07_1 for a detailed explanation.
    cache_path: str = CACHE_PATH

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        cache = json.load(open(self.cache_path)) if os.path.exists(self.cache_path) else {}
        texts_to_embed = [t for t in texts if t not in cache]
        if texts_to_embed:
            print(f"Embedding {len(texts_to_embed)} new text(s) via the API...")
            for text, vector in zip(texts_to_embed, super().embed_documents(texts_to_embed)):
                cache[text] = vector
            json.dump(cache, open(self.cache_path, "w"))
        else:
            print("All texts already in cache, no API call needed.")
        return [cache[t] for t in texts]


embeddings = CachedOllamaEmbeddings(model=EMBEDDING_MODEL, base_url=LLM_URL)
vectorstore = InMemoryVectorStore(embeddings)
vectorstore.add_documents(docs)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

All texts already in cache, no API call needed.


## Why build a raw graph?

`create_agent` (used in `06_2_agents.ipynb` and `07_2_agentic_rag.ipynb`) is actually built on top of **LangGraph** already — it just hides the graph from you behind a simple `.invoke()` call. Here, we build the graph ourselves with LangGraph's `StateGraph`, to get full, explicit control over the flow.

Three concepts:

- **State** — a shared data structure (we'll use a `TypedDict`) that flows through every step of the graph. Each node receives the current state and returns a partial update to it.
- **Nodes** — plain Python functions, each doing one step of work (retrieve, grade, rewrite, generate, ...).
- **Edges** — connections between nodes. A normal edge always goes A → B. A **conditional edge** picks the next node at runtime, based on a routing function's return value — this is how the self-correction loop is implemented.

In [15]:
from typing import TypedDict
from langchain_core.documents import Document as LCDocument


class GraphState(TypedDict):
    question: str
    documents: list[LCDocument]
    generation: str
    retries: int


MAX_RETRIES = 2  # safety valve so a genuinely unanswerable question doesn't loop forever

## Node 1: Retrieve

The simplest node — just calls the retriever we already built, with whatever the current `question` in the state is (which might be the original question, or a rewritten one, if we've looped back).

In [16]:
def retrieve(state: GraphState) -> dict:
    documents = retriever.invoke(state["question"])
    return {"documents": documents}

## Node 2: Grade documents (retrieval validation)

For each retrieved tweet, we ask the LLM a yes/no question: *is this actually relevant to the question?* We use structured output (`with_structured_output`, Pydantic models — see `04_1_pydantic_basics.ipynb`) to force a clean, parseable verdict instead of free-form text.

Two things we found by testing this against the real model before writing it here:

1. Asking directly for just a `binary_score` gave inconsistent, close-to-random results. Adding a `reasoning` field *before* `binary_score` in the schema — forcing the model to write out a short justification first — made the grading far more reliable. This works like a lightweight, forced chain-of-thought, even with the model's own extended thinking turned off.
2. We therefore set `reasoning=False` on this call: the model's default extended thinking mode is unnecessary overhead here (and, as noted in `06_2_agents.ipynb`, can even hang with grammar-constrained structured output) — the `reasoning` field in our schema does the same job faster.

In [17]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import Literal

llm = ChatOllama(model=LLM_REASONING, base_url=LLM_URL, temperature=0, reasoning=False)


class GradeDocuments(BaseModel):
    reasoning: str = Field(description="One brief sentence on why the tweet is or isn't relevant")
    binary_score: Literal["yes", "no"] = Field(description="'yes' if the tweet is relevant to the question, else 'no'")


grader = llm.with_structured_output(GradeDocuments)

GRADE_DOC_PROMPT = """You are a grader assessing the relevance of a retrieved customer tweet to a user question.

Retrieved tweet:
{document}

User question: {question}

Give a binary score 'yes' or 'no' to indicate whether the tweet is relevant to the question."""


def grade_documents(state: GraphState) -> dict:
    relevant_docs = []
    for doc in state["documents"]:
        result = grader.invoke(GRADE_DOC_PROMPT.format(document=doc.page_content, question=state["question"]))
        print(f"    grading {doc.page_content[:60]!r}: {result.binary_score} ({result.reasoning})")
        if result.binary_score == "yes":
            relevant_docs.append(doc)
    return {"documents": relevant_docs}

## Node 3: Rewrite question (query enhancement)

If none of the retrieved tweets were graded relevant, the question itself might be the problem — too vague, or phrased with different words than the knowledge base uses. We ask the LLM to reformulate it, then loop back to `retrieve`.

In [18]:
REWRITE_PROMPT = """Look at the input and try to reason about the underlying semantic intent.

Initial question:
{question}

Formulate an improved question, suitable for searching a database of real customer tweets about airlines.
Respond with only the improved question, nothing else."""


def rewrite_question(state: GraphState) -> dict:
    new_question = llm.invoke(REWRITE_PROMPT.format(question=state["question"])).content.strip()
    print(f"    rewriting: {state['question']!r} -> {new_question!r}")
    return {"question": new_question, "retries": state["retries"] + 1}

## Node 4: Generate answer

Same shape as the chain from `07_1_two_step_rag.ipynb`, just as a graph node instead of an LCEL chain. If no relevant tweets survived grading (e.g. we hit `MAX_RETRIES`), it's given an empty context and instructed to say so rather than guessing.

In [19]:
from langchain_core.prompts import ChatPromptTemplate

ANSWER_PROMPT = ChatPromptTemplate.from_template("""Answer the question based only on the following customer tweets. If the tweets are insufficient to answer, say you don't have enough information.

Tweets:
{context}

Question: {question}

Answer:""")


def generate_answer(state: GraphState) -> dict:
    context = "\n\n".join(d.page_content for d in state["documents"]) or "(no relevant tweets found)"
    chain = ANSWER_PROMPT | llm
    result = chain.invoke({"context": context, "question": state["question"]})
    return {"generation": result.content}

## Conditional edges: deciding what happens next

Two routing decisions drive the self-correction loop:

- **After grading**: if at least one relevant tweet survived, move on to `generate_answer`. If none did, and we haven't exhausted our retries, go to `rewrite_question`. If we *have* exhausted retries, generate anyway (with the LLM instructed to admit it doesn't know) rather than looping forever.
- **After generating**: grade the answer itself (answer validation) — does it actually address the question? If not, and retries remain, go back and rewrite the question. Otherwise, stop.

A routing function is just a plain function that returns a string; `add_conditional_edges` maps those strings to the next node.

In [20]:
class GradeAnswer(BaseModel):
    reasoning: str = Field(description="One brief sentence on whether the answer addresses the question")
    binary_score: Literal["yes", "no"] = Field(description="'yes' if the answer addresses the question, else 'no'")


answer_grader = llm.with_structured_output(GradeAnswer)

GRADE_ANSWER_PROMPT = """You are a grader assessing whether an answer actually addresses a user's question.

Question: {question}
Answer: {generation}

Give a binary score 'yes' or 'no' to indicate whether the answer addresses the question."""


def decide_after_grading(state: GraphState) -> str:
    if state["documents"]:
        return "generate"
    if state["retries"] >= MAX_RETRIES:
        print("    max retries reached, generating anyway")
        return "generate"
    return "rewrite"


def decide_after_answer(state: GraphState) -> str:
    if state["retries"] >= MAX_RETRIES:
        return "end"
    result = answer_grader.invoke(GRADE_ANSWER_PROMPT.format(question=state["question"], generation=state["generation"]))
    print(f"    grading answer: {result.binary_score} ({result.reasoning})")
    return "end" if result.binary_score == "yes" else "rewrite"

## Assembling the graph

In [21]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(GraphState)
builder.add_node("retrieve", retrieve)
builder.add_node("grade_documents", grade_documents)
builder.add_node("rewrite_question", rewrite_question)
builder.add_node("generate_answer", generate_answer)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade_documents")
builder.add_conditional_edges(
    "grade_documents", decide_after_grading, {"generate": "generate_answer", "rewrite": "rewrite_question"}
)
builder.add_edge("rewrite_question", "retrieve")
builder.add_conditional_edges(
    "generate_answer", decide_after_answer, {"end": END, "rewrite": "rewrite_question"}
)

app = builder.compile()

# A text (Mermaid) representation of the graph - paste it into https://mermaid.live to see it visually
print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade_documents(grade_documents)
	rewrite_question(rewrite_question)
	generate_answer(generate_answer)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	generate_answer -. &nbsp;end&nbsp; .-> __end__;
	generate_answer -. &nbsp;rewrite&nbsp; .-> rewrite_question;
	grade_documents -. &nbsp;generate&nbsp; .-> generate_answer;
	grade_documents -. &nbsp;rewrite&nbsp; .-> rewrite_question;
	retrieve --> grade_documents;
	rewrite_question --> retrieve;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## Running the graph

**Case 1: a clear, well-answerable question.** The first retrieval should already contain relevant tweets, so we expect no rewrite loop — straight through to a generated, validated answer.

In [22]:
result = app.invoke({"question": "What do customers say about lost luggage?", "documents": [], "generation": "", "retries": 0})
print("\nFINAL ANSWER:", result["generation"])

All texts already in cache, no API call needed.
    grading '@united still missing my luggage, was promised someone would': yes (The retrieved tweet specifically describes a customer's experience with missing/lost luggage (the flight from Shanghai to DC via ORD), which directly answers the user's question about what customers say regarding lost luggage.)
    grading "@united what's the point of offering a free checked bag of y": yes (The tweet specifically addresses the issue of lost checked bags (lost luggage) and expresses a customer's frustration regarding it, which directly answers the user's question about what customers say about lost luggage.)
    grading "@United has time to respond to everyone else's complaints bu": no (The retrieved tweet discusses a general complaint about poor customer service and the loss of a customer, but it does not mention lost luggage or any specific issues regarding baggage.)
    grading answer: yes (The answer directly provides specific examples of 

**Case 2: a question entirely outside the knowledge base.** Our corpus is customer tweets about their flying experience — ask a technical question far outside that, and every retrieved tweet should get graded irrelevant, triggering the rewrite loop, until `MAX_RETRIES` is hit and the graph gives up *gracefully* instead of hallucinating an answer.

In [23]:
result = app.invoke({"question": "What are people saying about airplane engine fuel efficiency standards?", "documents": [], "generation": "", "retries": 0})
print("\nFINAL ANSWER:", result["generation"])

All texts already in cache, no API call needed.
    grading '@AmericanAir research buddy. Read my previous tweets and get': no (The retrieved tweet is a defensive response to an airline account, telling them to look at previous context. It does not contain any information regarding airplane engine fuel efficiency standards.)
    grading "@JetBlue yesterday's deal?": no (The retrieved tweet asks about a specific deal from yesterday related to JetBlue, which does not mention or relate to airplane engine fuel efficiency standards.)
    grading "@JetBlue's CEO battles to appease passengers and Wall Street": no (The retrieved tweet discusses JetBlue's CEO dealing with passenger and Wall Street concerns, but it does not mention airplane engine fuel efficiency standards.)
    rewriting: 'What are people saying about airplane engine fuel efficiency standards?' -> '"airplane engine fuel efficiency" OR "fuel efficiency standards" OR "engine fuel consumption" airline OR flight'
All texts already 

Notice the difference from `07_1_two_step_rag.ipynb`: 2-Step RAG would have silently stuffed three irrelevant tweets into the prompt and hoped the LLM ignored them. Here, the graph *knows* it couldn't find anything relevant — it tried to fix the query, failed, and said so honestly instead of guessing. That's the whole point of the validation steps.

## Choosing an architecture

| | 2-Step RAG | Agentic RAG | Hybrid RAG |
|---|---|---|---|
| Retrieval | always | LLM decides | always, but validated |
| Predictability | high | lower (LLM controls flow) | high (explicit graph) |
| Latency / LLM calls | lowest | variable | highest (grading + possible retries) |
| Can self-correct a bad retrieval? | no | no | yes |
| Good for | simple, narrow-domain Q&A where retrieval is (almost) always needed | assistants that field a mix of general and domain-specific questions | high-stakes Q&A where grounded, validated answers matter more than latency |

There's no universally "best" one — pick based on how much you need to trust the answer versus how much latency/cost you can afford.

## Exercise: Add a retry counter to the printed trace

Modify `generate_answer` (or add a small wrapper around `app.invoke`) so that when the graph finishes, it also reports how many rewrite retries were used before it stopped. Test it on both example questions above and compare.

In [24]:
# Insert code here...

<details>
<summary><b>Show solution</b></summary>

```python
def run_with_report(question):
    result = app.invoke({"question": question, "documents": [], "generation": "", "retries": 0})
    print(f"Retries used: {result['retries']} / {MAX_RETRIES}")
    print(f"Final answer: {result['generation']}")
    return result


run_with_report("What do customers say about lost luggage?")
run_with_report("What are people saying about airplane engine fuel efficiency standards?")
```

</details>